In [2]:
import os
import pandas as pd

# Зчитуємо всі csv файли з папок та об'єднуємо в один датафрейм
dfs = []

for label in ['idle', 'running', 'stairs', 'walking']:
    folder = f'data/{label}'
    for file in os.listdir(folder):
        df_mini = pd.read_csv(f'{folder}/{file}')
        df_mini['label'] = label
        dfs.append(df_mini)

df = pd.concat(dfs, ignore_index=True)
df

,accelerometer_X,accelerometer_Y,accelerometer_Z,label
0,0.009577,5.937620,7.570466,idle
1,0.086191,6.555324,7.785944,idle
2,0.004788,6.440403,7.139510,idle
3,0.277727,6.430826,7.690176,idle
4,-0.047884,6.239290,7.340623,idle
...,...,...,...,...
193855,-1.877054,-9.428367,-2.499546,walking
193856,-10.093954,-13.819332,-1.192312,walking
193857,-0.919373,-2.030283,22.634783,walking
193858,0.201113,-12.531252,0.383072,walking


In [3]:
# Рахуємо скільки представників в кожному класі
df['label'].value_counts()

label
running    102240
walking     55500
idle        31170
stairs       4950
Name: count, dtype: int64

In [4]:
# Дивимося чи є пропуски
df.isnull().sum()

accelerometer_X    0
accelerometer_Y    0
accelerometer_Z    0
label              0
dtype: int64

In [11]:
from scipy.stats import kurtosis
import numpy as np



WINDOW_SIZE = 50
axes = ['accelerometer_X', 'accelerometer_Y', 'accelerometer_Z']

def rms(x):
    return np.sqrt((x**2).mean())

# Нумеруємо вікна — кожні 50 рядків одного класу отримують однаковий window_id
df['window_id'] = df.groupby('label').cumcount() // WINDOW_SIZE

# Набір A — базові статистики (mean, std, min, max)
df_features_a = df.groupby(['label', 'window_id'])[axes].agg(
    ['mean', 'std', 'min', 'max']
).reset_index()
df_features_a.columns = ['label', 'window_id'] + [
    f'{ax}_{stat}' for ax in axes for stat in ['mean', 'std', 'min', 'max']
]
df_features_a = df_features_a.drop(columns='window_id')

# Набір B — додаткові статистики для порівняння з набором A
df_features_b = df.groupby(['label', 'window_id'])[axes].agg(
    ['median', 'skew', kurtosis, rms]
).reset_index()
df_features_b.columns = ['label', 'window_id'] + [
    f'{ax}_{stat}' for ax in axes for stat in ['median', 'skew', 'kurtosis', 'rms']
]
df_features_b = df_features_b.drop(columns='window_id')

df_features_a.shape, df_features_b.shape

((3878, 13), (3878, 13))

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Набір A
X_a = df_features_a.drop(columns='label')
y_a = df_features_a['label']

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_a, y_a, test_size=0.2, random_state=42
)

# Набір B
X_b = df_features_b.drop(columns='label')
y_b = df_features_b['label']

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y_b, test_size=0.2, random_state=42
)

# Нормалізація
scaler_a = StandardScaler()
X_train_a = scaler_a.fit_transform(X_train_a)
X_test_a = scaler_a.transform(X_test_a)

scaler_b = StandardScaler()
X_train_b = scaler_b.fit_transform(X_train_b)
X_test_b = scaler_b.transform(X_test_b)

In [14]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# SVM
svm_a = SVC(random_state=42)
svm_a.fit(X_train_a, y_train_a)

svm_b = SVC(random_state=42)
svm_b.fit(X_train_b, y_train_b)

# Random Forest
rf_a = RandomForestClassifier(random_state=42)
rf_a.fit(X_train_a, y_train_a)

rf_b = RandomForestClassifier(random_state=42)
rf_b.fit(X_train_b, y_train_b)

print("Всі моделі навчені")

Всі моделі навчені!


In [15]:
from sklearn.metrics import classification_report

print("=== SVM на наборі A ===")
print(classification_report(y_test_a, svm_a.predict(X_test_a)))

print("=== SVM на наборі B ===")
print(classification_report(y_test_b, svm_b.predict(X_test_b)))

print("=== Random Forest на наборі A ===")
print(classification_report(y_test_a, rf_a.predict(X_test_a)))

print("=== Random Forest на наборі B ===")
print(classification_report(y_test_b, rf_b.predict(X_test_b)))

=== SVM на наборі A ===
              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       143
     running       1.00      1.00      1.00       389
      stairs       0.92      0.96      0.94        23
     walking       1.00      0.99      0.99       221

    accuracy                           1.00       776
   macro avg       0.98      0.99      0.98       776
weighted avg       1.00      1.00      1.00       776

=== SVM на наборі B ===
              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       143
     running       1.00      1.00      1.00       389
      stairs       1.00      0.91      0.95        23
     walking       0.99      1.00      1.00       221

    accuracy                           1.00       776
   macro avg       1.00      0.98      0.99       776
weighted avg       1.00      1.00      1.00       776

=== Random Forest на наборі A ===
              precision    recall  f1-score   su

## Висновки

### Дані
Датасет містить дані акселерометра з 4 класів активності: idle, running, stairs, walking.
Помітний дисбаланс класів — stairs має лише 4950 записів проти 102240 у running.

### Результати

| Модель | Набір фічів | Accuracy | F1 stairs |
|---|---|---|---|
| SVM | A (mean, std, min, max) | 1.00 | 0.94 |
| SVM | B (median, skew, kurtosis, rms) | 1.00 | 0.95 |
| Random Forest | A | 1.00 | 1.00 |
| Random Forest | B | 1.00 | 0.98 |

### Порівняння моделей
- **Random Forest** показав кращий результат на класі stairs — найскладнішому через дисбаланс даних
- **SVM** дещо гірше справляється з малопредставленими класами

### Порівняння наборів фічів
Базові фічі (набір A) виявились достатніми — додаткові статистики (набір B) суттєвого покращення не дали

### Загальний висновок
Для класифікації активності за даними акселерометра достатньо базових time domain features.
Random Forest є кращим вибором ніж SVM для даної задачі, особливо враховуючи дисбаланс класів.